In [1]:
# Instalación de librerías para RAG, bases vectoriales e ingesta de datos
!pip install -q langchain langchain-community qdrant-client sentence-transformers pandas openpyxl pypdf
print("Entorno preparado con éxito.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Entorno preparado con éxito.


In [2]:
!pip install -q python-dotenv

import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# 1. Intentar cargar desde archivo .env (estándar local exigido en la rúbrica)
load_dotenv()
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

# 2. Si no hay archivo .env, leer desde los Secrets seguros de Google Colab
if not QDRANT_URL or not QDRANT_API_KEY:
    from google.colab import userdata
    QDRANT_URL = userdata.get('QDRANT_URL')
    QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')

# Conexión al clúster sin exponer credenciales en el código ni en los outputs
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY
)

nombre_coleccion = "retail_vector_store"

colecciones = [col.name for col in client.get_collections().collections]
if nombre_coleccion not in colecciones:
    client.create_collection(
        collection_name=nombre_coleccion,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )
    print(f"Colección '{nombre_coleccion}' creada exitosamente en Qdrant Cloud.")
else:
    print(f"Conexión verificada. Colección '{nombre_coleccion}' activa y lista.")

Conexión verificada. Colección 'retail_vector_store' activa y lista.


In [4]:
import os
import urllib.request
import pandas as pd

archivo_excel = "Online Retail.xlsx"
url_uci_oficial = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

# 1. Verificar si el archivo existe en el entorno; si no, descargarlo automáticamente de la UCI
if not os.path.exists(archivo_excel):
    print("El archivo no se encontró localmente. Descargando 'Online Retail.xlsx' desde el repositorio oficial de la UCI...")
    try:
        urllib.request.urlretrieve(url_uci_oficial, archivo_excel)
        print("¡Descarga completada con éxito! El archivo quedó guardado en el entorno.")
    except Exception as e:
        print(f"Error al descargar desde la UCI: {e}")

# 2. Lectura y limpieza de datos estructurados
if os.path.exists(archivo_excel):
    # Leemos las primeras 300 filas para optimizar el consumo de la capa gratuita en Qdrant Cloud
    df_ventas = pd.read_excel(archivo_excel, nrows=300)

    # Limpieza de registros nulos en campos críticos y filtrado de valores válidos
    df_ventas = df_ventas.dropna(subset=['Description', 'InvoiceNo', 'StockCode']).copy()
    df_ventas['InvoiceNo'] = df_ventas['InvoiceNo'].astype(str)
    df_ventas = df_ventas[~df_ventas['InvoiceNo'].str.startswith('C')]  # Excluir facturas canceladas
    df_ventas = df_ventas[(df_ventas['Quantity'] > 0) & (df_ventas['UnitPrice'] > 0)]

    print(f"Dataset estructurado cargado y depurado: {len(df_ventas)} registros válidos listos para indexar.")
else:
    # Respaldo en memoria por si el entorno no tiene salida a internet hacia la UCI
    print("Cargando muestra demostrativa de respaldo...")
    df_ventas = pd.DataFrame({
        "InvoiceNo": ["536365", "536366", "536367", "536368", "536369"],
        "StockCode": ["85123A", "71053", "84406B", "84029G", "21730"],
        "Description": [
            "WHITE HANGING HEART T-LIGHT HOLDER",
            "WHITE METAL LANTERN",
            "CREAM CUPID HEARTS COAT HANGER",
            "KNITTED UNION FLAG HOT WATER BOTTLE",
            "GLASS STAR FROSTED T-LIGHT HOLDER"
        ],
        "Quantity": [6, 6, 8, 6, 6],
        "UnitPrice": [2.55, 3.39, 2.75, 3.39, 4.25],
        "CustomerID": [17850, 17850, 13047, 13047, 17850],
        "Country": ["United Kingdom", "United Kingdom", "United Kingdom", "United Kingdom", "United Kingdom"]
    })

# 3. Mostrar vista previa de control en el Notebook
print("\n--- Vista previa de los registros estructurados a vectorizar ---")
display(df_ventas[['InvoiceNo', 'StockCode', 'Description', 'UnitPrice', 'CustomerID', 'Country']].head())

El archivo no se encontró localmente. Descargando 'Online Retail.xlsx' desde el repositorio oficial de la UCI...
¡Descarga completada con éxito! El archivo quedó guardado en el entorno.
Dataset estructurado cargado y depurado: 291 registros válidos listos para indexar.

--- Vista previa de los registros estructurados a vectorizar ---


,InvoiceNo,StockCode,Description,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,3.39,17850,United Kingdom


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# 1. Crear el documento complementario obligatorio (Políticas y SLA de E-commerce)
politicas_comerciales = """
POLÍTICA INSTITUCIONAL DE GARANTÍAS, DEVOLUCIONES Y ATENCIÓN E-COMMERCE
1. Derecho de Retracto y Devolución: El comprador dispone de 30 días calendario tras la recepción del pedido para solicitar el cambio o reembolso.
2. Estado Requerido del Producto: El artículo debe entregarse en su empaque original, sin señales de desgaste, con manuales y precintos intactos.
3. Cobertura Técnica: Los productos de decoración metálica, iluminación, candelabros y accesorios de hogar tienen una garantía directa de fábrica de 180 días calendario.
4. Tiempos de Entrega (SLA Logístico): Los envíos estándar a nivel nacional demoran de 2 a 5 días hábiles. Las ciudades principales cuentan con despacho prioritario de 24 a 48 horas.
5. Reclamaciones por Avería en Tránsito: Todo reporte de daño o rotura durante el envío debe presentarse en las primeras 48 horas adjuntando evidencia fotográfica y número de factura.
"""

archivo_politicas = "politicas_garantia_retail.txt"
with open(archivo_politicas, "w", encoding="utf-8") as f:
    f.write(politicas_comerciales.strip())

# 2. Cargar el documento y dividirlo en fragmentos (chunks)
cargador = TextLoader(archivo_politicas, encoding="utf-8")
documentos = cargador.load()

fragmentador = RecursiveCharacterTextSplitter(
    chunk_size=280,
    chunk_overlap=40,
    separators=["\n\n", "\n", ".", " "]
)
fragmentos = fragmentador.split_documents(documentos)

print(f"✅ Corpus no estructurado procesado: {len(fragmentos)} fragmentos (chunks) generados.")
for i, frag in enumerate(fragmentos, 1):
    print(f"  Chunk {i}: {frag.page_content[:75]}...")

✅ Corpus no estructurado procesado: 5 fragmentos (chunks) generados.
  Chunk 1: POLÍTICA INSTITUCIONAL DE GARANTÍAS, DEVOLUCIONES Y ATENCIÓN E-COMMERCE
1. ...
  Chunk 2: 2. Estado Requerido del Producto: El artículo debe entregarse en su empaque...
  Chunk 3: 3. Cobertura Técnica: Los productos de decoración metálica, iluminación, ca...
  Chunk 4: 4. Tiempos de Entrega (SLA Logístico): Los envíos estándar a nivel nacional...
  Chunk 5: 5. Reclamaciones por Avería en Tránsito: Todo reporte de daño o rotura dura...


In [8]:
import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from qdrant_client.models import PointStruct, Distance, VectorParams

# 1. Inicializar el modelo de embeddings (384 dimensiones)
modelo_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Asegurar que la variable de nombre de colección esté definida y activa en Qdrant
if 'nombre_coleccion' not in locals():
    nombre_coleccion = "retail_vector_store"

colecciones_activas = [c.name for c in client.get_collections().collections]
if nombre_coleccion not in colecciones_activas:
    client.create_collection(
        collection_name=nombre_coleccion,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE)
    )

textos_a_vectorizar = []
payloads_temporales = []

# 2. Preparar los fragmentos del documento no estructurado (Políticas)
for fragmento in fragmentos:
    texto_limpio = str(fragmento.page_content).strip()
    textos_a_vectorizar.append(texto_limpio)
    payloads_temporales.append({
        "tipo_fuente": "no_estructurado_politicas",
        "contenido": texto_limpio,
        "categoria": "Políticas y Garantías",
        "pilar": "Ventas"
    })

# 3. Preparar los registros del dataset estructurado (df_ventas de la Celda 3)
for _, fila in df_ventas.iterrows():
    descripcion = str(fila.get('Description', 'Sin descripción')).strip()
    sku = str(fila.get('StockCode', 'N/A')).strip()
    factura = str(fila.get('InvoiceNo', 'N/A')).strip()
    pais = str(fila.get('Country', 'N/A')).strip()

    # Manejo seguro de valores nulos o formatos en Precio y CustomerID
    precio = float(fila['UnitPrice']) if pd.notna(fila.get('UnitPrice')) else 0.0
    cliente_id = str(int(float(fila['CustomerID']))) if pd.notna(fila.get('CustomerID')) else "Anónimo"

    texto_descriptivo = (
        f"Producto en catálogo: {descripcion}. "
        f"Código SKU: {sku}. Precio unitario: {precio:.2f} USD. País: {pais}."
    )

    textos_a_vectorizar.append(texto_descriptivo)
    payloads_temporales.append({
        "tipo_fuente": "estructurado_excel",
        "contenido": texto_descriptivo,
        "InvoiceNo": factura,
        "StockCode": sku,
        "UnitPrice": precio,
        "CustomerID": cliente_id,
        "Country": pais,
        "pilar": "Ventas"
    })

# 4. Generar todos los embeddings en un solo bloque (mucho más rápido y estable)
print(f"Generando embeddings para {len(textos_a_vectorizar)} registros...")
vectores_generados = modelo_embeddings.embed_documents(textos_a_vectorizar)

# 5. Construir los PointStruct y subir por lotes de 100 a Qdrant Cloud
lote_puntos = [
    PointStruct(id=idx + 1, vector=vec, payload=pay)
    for idx, (vec, pay) in enumerate(zip(vectores_generados, payloads_temporales))
]

tamano_lote = 100
for i in range(0, len(lote_puntos), tamano_lote):
    client.upsert(
        collection_name=nombre_coleccion,
        points=lote_puntos[i:i + tamano_lote]
    )

print(f"✅ ¡Carga masiva exitosa! Se indexaron {len(lote_puntos)} vectores en la colección '{nombre_coleccion}'.")

/tmp/ipykernel_845/3658038133.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  modelo_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generando embeddings para 296 registros...
✅ ¡Carga masiva exitosa! Se indexaron 296 vectores en la colección 'retail_vector_store'.


In [9]:
consultas_evaluacion = [
    "¿Cuál es el plazo máximo para devolver un producto y en qué condiciones debe estar?",
    "Busco un farol o linterna metálica blanca decorativa para el hogar",
    "¿Qué debo hacer si mi pedido llegó roto durante el transporte?"
]

for pregunta in consultas_evaluacion:
    print("\n" + "="*70)
    print(f"PREGUNTA: {pregunta}")
    print("="*70)

    vector_pregunta = modelo_embeddings.embed_query(pregunta)
    recuperados = client.query_points(
        collection_name=nombre_coleccion,
        query=vector_pregunta,
        limit=2
    )

    for pos, punto in enumerate(recuperados.points, 1):
        print(f"[{pos}] Score Similitud Coseno: {punto.score:.4f} | Origen: {punto.payload.get('tipo_fuente')}")
        print(f"    Texto: {punto.payload.get('contenido')}")
        if punto.payload.get('tipo_fuente') == 'estructurado_excel':
            print(f"    Metadatos: SKU={punto.payload.get('StockCode')} | Precio={punto.payload.get('UnitPrice')} USD")


PREGUNTA: ¿Cuál es el plazo máximo para devolver un producto y en qué condiciones debe estar?
[1] Score Similitud Coseno: 0.4850 | Origen: no_estructurado_politicas
    Texto: 2. Estado Requerido del Producto: El artículo debe entregarse en su empaque original, sin señales de desgaste, con manuales y precintos intactos.
[2] Score Similitud Coseno: 0.4524 | Origen: no_estructurado_politicas
    Texto: POLÍTICA INSTITUCIONAL DE GARANTÍAS, DEVOLUCIONES Y ATENCIÓN E-COMMERCE
1. Derecho de Retracto y Devolución: El comprador dispone de 30 días calendario tras la recepción del pedido para solicitar el cambio o reembolso.

PREGUNTA: Busco un farol o linterna metálica blanca decorativa para el hogar
[1] Score Similitud Coseno: 0.5829 | Origen: no_estructurado_politicas
    Texto: 3. Cobertura Técnica: Los productos de decoración metálica, iluminación, candelabros y accesorios de hogar tienen una garantía directa de fábrica de 180 días calendario.
[2] Score Similitud Coseno: 0.4499 | Origen: n